In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

In [2]:
# choose a ticker and sort by date
def Day20Return(market_data, ticker, eventDate):
    df = market_data[market_data['Ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    match = df.index[pd.to_datetime(df['Date'], utc = True).dt.strftime('%Y-%m-%d') == pd.to_datetime(eventDate, utc = True).strftime('%Y-%m-%d')]
    if len(match)==0:
        return np.nan
    idx = match[0]

    # check data avlb
    if idx < 90 or (idx+20) > len(df):
        return np.nan

    # gather data from -90 days prior to +20 days post event
    pre_data = df.iloc[idx-90 : idx][['Close', 'Volume', 'NBI']].values
    post_data = df.iloc[idx : idx+20]['Close'].values
    norm = pre_data/pre_data[0]

    if (pre_data[0] == 0).any():
        return np.nan

    if np.isnan(norm).any() or np.isinf(norm).any() or np.isnan(post_data).any():
        return np.nan

    # sliding window 30d input, 20d outcome
    X = [norm[i : i+30].flatten() for i in range (41)]
    y = [norm[i+30 : i+50, 0] for i in range (41)]

    # fit regression
    pred = Ridge().fit(X,y).predict([norm[-30:].flatten()])[0] * pre_data[0,0]
    return (post_data - pred).sum() / pred.sum()

In [3]:
market_data_df = pd.read_csv("datasets/market_ds.csv")

In [4]:
Day20Return(market_data_df, 'AMGN', '2025-03-30')

nan

In [5]:
catalyst_df = pd.read_csv("datasets/catalyst_ds.csv")

In [6]:
catalyst_df['NCAR20'] = [Day20Return(market_data_df, ticker, date) 
                         for ticker, date in zip(catalyst_df['company_ticker'], 
                                                             pd.to_datetime(catalyst_df['date'], utc = True).dt.strftime('%Y-%m-%d'))]

/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: divide by zero encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: invalid value encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: divide by zero encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: invalid value encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: divide by zero encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/35/mk5rptzn3yqgvrbrxm07ls340000gn/T/ipykernel_34497/188869443.py:16: RuntimeWarning: invalid value encountered in divide
  norm = pre_data/pre_data[0]
/var/folders/

In [7]:
catalyst_df_filtered = catalyst_df[(catalyst_df['date'] >= '2025-01-01') & (catalyst_df['date'] <= '2026-06-30')]

In [8]:
catalyst_df_filtered.to_csv("datasets/catalyst_NCAR20.csv", index = False)

In [9]:
catalyst_df_filtered = pd.read_csv("datasets/catalyst_NCAR20.csv")
company_features_df = pd.read_csv("datasets/company_ds.csv")

In [10]:
final_df = pd.merge(
    catalyst_df_filtered,
    company_features_df,
    left_on='company_ticker',
    right_on='ticker',
    how = 'left',
).drop(columns=['ticker'])

In [11]:
final_df.head()

,catalyst,date,company_name,company_ticker,drug_name,nct_number,label,press_link,stage,polarity,NCAR20,Total Revenue,Cost Of Revenue,Operating Revenue,Cash And Cash Equivalents,Total Debt,Capital Expenditure,drug_count
0,Phase 1 trial recently initiated noted July 17...,2025-07-17,Praxis Precision Medicines Inc.,PRAX,Relutrigine (PRAX-562) - (EMERALD),NCT07010471,Phase 1,https://www.biopharmcatalyst.com/company/PRAX/...,Phase 1,0.032329,0.126450,0.0,NaN,0.0,357329000.0,110000.0,-56000.0,4
1,Phase 1 data published in the peer-reviewed jo...,2025-03-07,Cerenome Inc Com Par $0.0001 (New) Jan 2026,PSTV,REYOBIQ (rhenium Re186 obisbemeda) - (ReSPECT-...,NCT01906385,Phase 1,https://www.biopharmcatalyst.com/company/PSTV/...,Phase 1,0.902388,NaN,0.0,NaN,0.0,4256000.0,821000.0,-67000.0,3
2,Phase 1 trial of CUE-101 showed an additional ...,2025-07-16,Cue Biopharma Inc.,CUE,CUE-101,NCT03978689,Phase 1,https://www.biopharmcatalyst.com/company/CUE/n...,Phase 1,0.784513,0.043199,27466000.0,NaN,27466000.0,27136000.0,4197000.0,-177000.0,2
3,Phase 1/1b data showed an overall response rat...,2025-02-14,Cellectis S.A.,CLLS,ALLO-501/501A + ALLO-647 - (ALPHA2),NCT04416984,Phase 1,https://www.biopharmcatalyst.com/company/ALLO/...,Phase 1,0.904840,0.586324,72949000.0,9797000.0,72949000.0,61533000.0,90733000.0,-3533000.0,4
4,Phase 1/1b data showed an overall response rat...,2025-02-14,Allogene Therapeutics Inc.,ALLO,ALLO-501/501A + ALLO-647 - (ALPHA2),NCT04416984,Phase 1,https://www.biopharmcatalyst.com/company/ALLO/...,Phase 1,0.904840,0.042479,0.0,NaN,0.0,51688000.0,83253000.0,-386000.0,3


In [12]:
final_df.to_csv("datasets/final_ds_NCAR20.csv", index = False)